# Supply Chain Intelligence — XGBoost Regression Models

## Predicting forward absolute returns over 5d and 20d horizons from rolling GDELT signals

This notebook trains XGBoost regressors for volatility-style targets, not directional return forecasts. The primary targets are `abs_return_5d_fwd` and `abs_return_20d_fwd`, which measure the magnitude of future commodity price moves.

The pipeline reads Spark-produced per-commodity feature parquet from HDFS through a WebHDFS/ngrok bridge, evaluates time-split regression quality, computes SHAP attributions, and writes dashboard-ready artifacts back to HDFS.

In [ ]:
# Setup
%pip install -q xgboost shap pyarrow seaborn scikit-learn hdfs

import io
import json
import os
import random
import subprocess
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlencode, urljoin, urlparse, urlunparse

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import shap
import xgboost as xgb
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from xgboost import XGBRegressor

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
random.seed(SEED)

try:
    gpu_probe = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    HAS_GPU = gpu_probe.returncode == 0
except FileNotFoundError:
    HAS_GPU = False

XGB_DEVICE = "cuda" if HAS_GPU else "cpu"
print(f"XGBoost version: {xgb.__version__}")
print(f"Detected XGBoost device: {XGB_DEVICE}")
if HAS_GPU:
    print(gpu_probe.stdout.splitlines()[0])
else:
    print("No GPU detected by nvidia-smi; notebook will run on CPU unless Colab runtime is changed.")

In [ ]:
# Connection setup: paste the public ngrok URL for HttpFS/WebHDFS.
NGROK_URL = "https://...ngrok-free.dev"  # no trailing slash
# The local Docker HDFS paths are owned by root; hadoop often cannot mkdir under /supply-chain.
HDFS_USER = "root"
REQUEST_HEADERS = {"ngrok-skip-browser-warning": "true"}
LOCAL_DATA_ROOT = Path("/content/data")
LOCAL_ARTIFACT_ROOT = Path("/content/artifacts")
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)


class HdfsBridge:
    """Small HttpFS client matching the working classification notebook pattern."""

    def __init__(self, webhdfs_url: str, user: str = "root"):
        from hdfs import InsecureClient

        self._client = InsecureClient(webhdfs_url.rstrip("/"), user=user)
        self.url = webhdfs_url.rstrip("/")
        self.user = user

    def mkdirs(self, hdfs_dir: str) -> None:
        self._client.makedirs(hdfs_dir)

    def write_bytes(self, hdfs_path: str, content: bytes, overwrite: bool = True) -> None:
        create_url = _webhdfs_url(hdfs_path, "CREATE", overwrite=str(overwrite).lower())
        first = _request("PUT", create_url, allow_redirects=False)
        location = first.headers.get("Location")
        if not location:
            raise RuntimeError(f"HttpFS CREATE did not return a redirect for {hdfs_path}")
        second = _request(
            "PUT",
            _rewrite_redirect_url(location),
            data=content,
            headers={"Content-Type": "application/octet-stream"},
        )
        if second.status_code not in (200, 201):
            raise RuntimeError(f"Unexpected HttpFS upload status {second.status_code} for {hdfs_path}")

    def write_text(self, hdfs_path: str, text: str, overwrite: bool = True) -> None:
        self.write_bytes(hdfs_path, text.encode("utf-8"), overwrite=overwrite)


hdfs = HdfsBridge(NGROK_URL, user=HDFS_USER)
print("HttpFS WebHDFS:", hdfs.url, "user=", hdfs.user)


def _base_url() -> str:
    if "..." in NGROK_URL:
        raise ValueError("Set NGROK_URL to your live ngrok HttpFS URL before running HDFS cells.")
    return NGROK_URL.rstrip("/")


def _webhdfs_url(hdfs_path: str, op: str, **params) -> str:
    clean_path = "/" + hdfs_path.strip("/")
    query = {"op": op, "user.name": HDFS_USER, **params}
    return f"{_base_url()}/webhdfs/v1{clean_path}?{urlencode(query)}"


def _rewrite_redirect_url(location: str) -> str:
    """Rewrite datanode redirects back through ngrok when WebHDFS returns 307."""
    if not location:
        raise ValueError("Missing WebHDFS redirect location")
    loc = urlparse(location)
    base = urlparse(_base_url())
    # Preserve the WebHDFS path/query but force the public ngrok scheme/host.
    return urlunparse((base.scheme, base.netloc, loc.path, loc.params, loc.query, loc.fragment))


def _raise_for_status_with_body(response: requests.Response) -> None:
    try:
        response.raise_for_status()
    except requests.HTTPError as exc:
        body = response.text[:2000] if response.text else "<empty response body>"
        raise requests.HTTPError(f"{exc}\nWebHDFS response body:\n{body}", response=response) from exc


def _request(method: str, url: str, **kwargs) -> requests.Response:
    headers = {**REQUEST_HEADERS, **kwargs.pop("headers", {})}
    response = requests.request(method, url, headers=headers, timeout=120, **kwargs)
    _raise_for_status_with_body(response)
    return response


def hdfs_mkdirs_v2(hdfs_dir: str) -> bool:
    url = _webhdfs_url(hdfs_dir, "MKDIRS")
    response = _request("PUT", url)
    payload = response.json()
    ok = bool(payload.get("boolean", False))
    print(f"MKDIRS {hdfs_dir}: {ok} user={HDFS_USER}")
    return ok


def hdfs_upload_v2(local_path: str | Path, hdfs_path: str, overwrite: bool = True) -> int:
    """Upload through HttpFS using the hdfs client, matching the working classification notebook."""
    local_path = Path(local_path)
    parent = "/" + "/".join(hdfs_path.strip("/").split("/")[:-1])
    hdfs.mkdirs(parent)
    content = local_path.read_bytes()
    hdfs.write_bytes(hdfs_path, content, overwrite=overwrite)
    size = len(content)
    print(f"UPLOAD {local_path} -> {hdfs_path} ({size:,} bytes) user={hdfs.user}")
    return size


def hdfs_download(hdfs_dir: str, local_dir: str | Path) -> list[Path]:
    """Download parquet part files from an HDFS directory to local Colab storage."""
    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)
    list_url = _webhdfs_url(hdfs_dir, "LISTSTATUS")
    listing = _request("GET", list_url).json()
    statuses = listing.get("FileStatuses", {}).get("FileStatus", [])
    part_files = [
        item for item in statuses
        if item.get("type") == "FILE"
        and (item.get("pathSuffix", "").endswith(".parquet") or item.get("pathSuffix", "").startswith("part-"))
    ]
    if not part_files:
        raise FileNotFoundError(f"No parquet part files found in {hdfs_dir}")

    downloaded = []
    for item in part_files:
        name = item["pathSuffix"]
        hdfs_file = f"{hdfs_dir.rstrip('/')}/{name}"
        open_url = _webhdfs_url(hdfs_file, "OPEN")
        first = requests.get(open_url, headers=REQUEST_HEADERS, allow_redirects=False, timeout=120)
        if first.status_code in (307, 308):
            data_url = _rewrite_redirect_url(first.headers["Location"])
            response = _request("GET", data_url)
            content = response.content
        else:
            _raise_for_status_with_body(first)
            content = first.content
        out_path = local_dir / name
        out_path.write_bytes(content)
        downloaded.append(out_path)
    print(f"Downloaded {len(downloaded)} parquet parts from {hdfs_dir} to {local_dir}")
    return downloaded


print("Connection helpers ready. Set NGROK_URL before running download/upload cells.")

In [ ]:
# Data load
COMMODITIES = ["brent", "wti", "copper", "gold", "wheat", "soybeans"]
CHOKEPOINTS = ["hormuz", "suez", "red_sea", "black_sea", "malacca", "taiwan", "panama", "chile"]
BASE_METRICS = ["event_sum", "goldstein_mean", "tone_mean"]
WINDOWS = [7, 14, 30, 90]
TARGET_COLUMNS = ["return_5d_fwd", "return_20d_fwd", "abs_return_5d_fwd", "abs_return_20d_fwd"]
GDELT_FEATURE_COLS = [
    f"{cp}_{metric}_{window}d"
    for cp in CHOKEPOINTS
    for window in WINDOWS
    for metric in BASE_METRICS
]

commodity_frames: dict[str, pd.DataFrame] = {}
for commodity in COMMODITIES:
    hdfs_dir = f"/supply-chain/features/baseline/{commodity}"
    local_dir = LOCAL_DATA_ROOT / commodity
    part_paths = hdfs_download(hdfs_dir, local_dir)
    df = pd.concat([pd.read_parquet(path) for path in part_paths], ignore_index=True)
    df["event_date"] = pd.to_datetime(df["event_date"])
    df = df.sort_values("event_date").reset_index(drop=True)

    missing_targets = sorted(set(TARGET_COLUMNS) - set(df.columns))
    missing_gdelt = sorted(set(GDELT_FEATURE_COLS) - set(df.columns))
    if missing_targets or missing_gdelt:
        raise ValueError(
            f"{commodity} schema mismatch: missing_targets={missing_targets}, "
            f"missing_gdelt_count={len(missing_gdelt)}"
        )
    commodity_frames[commodity] = df
    print(
        f"{commodity:8s} rows={len(df):4d} "
        f"date={df['event_date'].min().date()}->{df['event_date'].max().date()} "
        f"columns={len(df.columns)}"
    )

print(f"Loaded {len(commodity_frames)} commodity feature tables.")

In [ ]:
# Feature/target setup
TARGETS = ["abs_return_5d_fwd", "abs_return_20d_fwd"]
REQUIRED_MACRO_COLS = ["treasury_10y", "usd_index"]
OPTIONAL_MACRO_COLS = ["vix"]

available_optional_macro = [
    col for col in OPTIONAL_MACRO_COLS
    if all(col in frame.columns for frame in commodity_frames.values())
]
MACRO_FEATURE_COLS = REQUIRED_MACRO_COLS + available_optional_macro
FEATURE_COLS = GDELT_FEATURE_COLS + ["volatility_20d"] + MACRO_FEATURE_COLS
EXPECTED_FEATURE_COUNT = 96 + 1 + len(MACRO_FEATURE_COLS)

for commodity, df in commodity_frames.items():
    missing_features = sorted(set(FEATURE_COLS) - set(df.columns))
    if missing_features:
        raise ValueError(f"{commodity} missing feature columns: {missing_features}")
    missing_targets = sorted(set(TARGETS) - set(df.columns))
    if missing_targets:
        raise ValueError(f"{commodity} missing target columns: {missing_targets}")

print(f"Targets: {TARGETS}")
print(f"Macro features: {MACRO_FEATURE_COLS}")
print(f"Feature count: {len(FEATURE_COLS)} (expected {EXPECTED_FEATURE_COUNT})")
if "vix" in MACRO_FEATURE_COLS:
    print("Using current 3-macro contract: treasury_10y, usd_index, vix.")
else:
    print("Using 2-macro contract because vix was not present in every commodity table.")

In [ ]:
# Time-based split: 70/15/15, no shuffle
splits: dict[tuple[str, str], dict[str, object]] = {}


def make_time_split(df: pd.DataFrame, target: str) -> dict[str, object]:
    work = df.dropna(subset=[target]).sort_values("event_date").reset_index(drop=True)
    n = len(work)
    if n < 30:
        raise ValueError(f"Not enough rows for {target}: {n}")
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)
    train = work.iloc[:train_end].copy()
    val = work.iloc[train_end:val_end].copy()
    test = work.iloc[val_end:].copy()
    return {
        "X_train": train[FEATURE_COLS],
        "y_train": train[target],
        "X_val": val[FEATURE_COLS],
        "y_val": val[target],
        "X_test": test[FEATURE_COLS],
        "y_test": test[target],
        "dates_test": test["event_date"].reset_index(drop=True),
        "train_df": train,
        "val_df": val,
        "test_df": test,
    }

summary_rows = []
for commodity, df in commodity_frames.items():
    for target in TARGETS:
        key = (commodity, target)
        split = make_time_split(df, target)
        splits[key] = split
        y = split["y_train"]
        summary_rows.append({
            "commodity": commodity,
            "target": target,
            "train_rows": len(split["y_train"]),
            "val_rows": len(split["y_val"]),
            "test_rows": len(split["y_test"]),
            "target_mean": y.mean(),
            "target_std": y.std(),
            "target_p50": y.quantile(0.50),
            "target_p90": y.quantile(0.90),
        })

split_summary_df = pd.DataFrame(summary_rows)
print("Time-split summary:")
display(split_summary_df)

In [ ]:
# Naive baseline computation: train-median constant predictor
naive_baselines: dict[tuple[str, str], dict[str, object]] = {}
baseline_rows = []

for key, split in splits.items():
    commodity, target = key
    naive_value = float(split["y_train"].median())
    y_test = split["y_test"].to_numpy()
    naive_pred = np.full_like(y_test, fill_value=naive_value, dtype=float)
    naive_mae = mean_absolute_error(y_test, naive_pred)
    naive_baselines[key] = {"naive_value": naive_value, "naive_test_mae": naive_mae}
    baseline_rows.append({
        "commodity": commodity,
        "target": target,
        "naive_value": naive_value,
        "naive_test_mae": naive_mae,
    })

baseline_df = pd.DataFrame(baseline_rows)
print("Naive test MAE baselines:")
display(baseline_df)

In [ ]:
# Single-commodity verification run: brent, abs_return_5d_fwd
param_dist = {
    "n_estimators":     [200, 400, 800],
    "max_depth":        [3, 4, 5, 6],
    "learning_rate":    [0.01, 0.03, 0.05, 0.1],
    "min_child_weight": [3, 5, 10],
    "subsample":        [0.7, 0.85, 1.0],
    "colsample_bytree": [0.5, 0.7, 1.0],
    "reg_lambda":       [1.0, 3.0, 10.0],
}


def build_base_regressor() -> XGBRegressor:
    return XGBRegressor(
        objective="reg:squarederror",
        eval_metric="mae",
        tree_method="hist",
        device=XGB_DEVICE,
        random_state=SEED,
        n_jobs=-1,
    )


def evaluate_regressor(model, split: dict[str, object], key: tuple[str, str]) -> dict[str, float]:
    y_val = split["y_val"].to_numpy()
    y_test = split["y_test"].to_numpy()
    pred_val = model.predict(split["X_val"])
    pred_test = model.predict(split["X_test"])
    naive_mae = naive_baselines[key]["naive_test_mae"]
    test_mae = mean_absolute_error(y_test, pred_test)
    rho = spearmanr(y_test, pred_test, nan_policy="omit").correlation
    return {
        "val_r2": r2_score(y_val, pred_val),
        "val_mae": mean_absolute_error(y_val, pred_val),
        "test_r2": r2_score(y_test, pred_test),
        "test_mae": test_mae,
        "naive_mae": naive_mae,
        "improvement_pct": (naive_mae - test_mae) / naive_mae * 100 if naive_mae else np.nan,
        "spearman": rho if pd.notna(rho) else np.nan,
    }

verification_key = ("brent", "abs_return_5d_fwd")
verification_split = splits[verification_key]
verification_search = RandomizedSearchCV(
    estimator=build_base_regressor(),
    param_distributions=param_dist,
    n_iter=30,
    scoring="neg_mean_absolute_error",
    cv=TimeSeriesSplit(n_splits=5),
    random_state=SEED,
    n_jobs=1,
    refit=True,
    verbose=1,
)
print(f"Starting verification search for {verification_key} on device={XGB_DEVICE}")
verification_search.fit(verification_split["X_train"], verification_split["y_train"])
verification_model = verification_search.best_estimator_
verification_metrics = evaluate_regressor(verification_model, verification_split, verification_key)
print("Best params:", verification_search.best_params_)
print("Verification metrics:")
print(json.dumps(verification_metrics, indent=2))
print("Verification gate complete; proceed to full loop if this cell finishes without error.")

In [ ]:
# Full loop over all 12 (commodity, target) pairs
results: dict[tuple[str, str], dict[str, object]] = {}
metrics_rows = []

for key, split in splits.items():
    commodity, target = key
    print(f"\n=== Training {commodity} / {target} ===")
    search = RandomizedSearchCV(
        estimator=build_base_regressor(),
        param_distributions=param_dist,
        n_iter=30,
        scoring="neg_mean_absolute_error",
        cv=TimeSeriesSplit(n_splits=5),
        random_state=SEED,
        n_jobs=1,
        refit=True,
        verbose=0,
    )
    search.fit(split["X_train"], split["y_train"])
    model = search.best_estimator_
    metrics = evaluate_regressor(model, split, key)
    pred_test = model.predict(split["X_test"])
    naive_value = naive_baselines[key]["naive_value"]
    predictions = pd.DataFrame({
        "event_date": split["dates_test"],
        "predicted_value": pred_test,
        "actual_value": split["y_test"].to_numpy(),
        "naive_value": naive_value,
    })
    results[key] = {
        "model": model,
        "search": search,
        "metrics": metrics,
        "predictions": predictions,
        "split": split,
    }
    row = {"commodity": commodity, "target": target, **metrics}
    row["decision_flag"] = "✓" if metrics["improvement_pct"] > 5 else ("✗" if metrics["test_r2"] < 0 else "")
    metrics_rows.append(row)
    print(json.dumps(metrics, indent=2))

metrics_df = pd.DataFrame(metrics_rows)
summary_cols = ["commodity", "target", "test_r2", "test_mae", "naive_mae", "improvement_pct", "spearman", "decision_flag"]
print("\nRegression summary table:")
display(metrics_df[summary_cols].sort_values(["target", "commodity"]).reset_index(drop=True))

In [ ]:
# DECISION GATE
from IPython.display import Markdown, display

criteria_md = """
## DECISION GATE

Model is usable for dashboard integration if:

(a) average test R² across all 12 models > 0.10, AND

(b) at least 6 of 12 models show `improvement_pct > 5%`.

If the verdict is `NOT_USABLE`, continue with SHAP analysis for interpretability, but the dashboard should not show model predictions.
"""
display(Markdown(criteria_md))

avg_test_r2 = float(metrics_df["test_r2"].mean())
models_improving = int((metrics_df["improvement_pct"] > 5).sum())
VERDICT = "USABLE" if (avg_test_r2 > 0.10 and models_improving >= 6) else "NOT_USABLE"
print(f"average_test_r2={avg_test_r2:.6f}")
print(f"models_with_improvement_gt_5pct={models_improving}/12")
print(f"DECISION_VERDICT={VERDICT}")

In [ ]:
# SHAP computation
shap_results: dict[tuple[str, str], dict[str, object]] = {}

for key, result in results.items():
    commodity, target = key
    model = result["model"]
    X_test = result["split"]["X_test"]
    print(f"Computing SHAP for {commodity} / {target}: X_test={X_test.shape}")
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    shap_values = np.asarray(shap_values)
    if shap_values.shape != X_test.shape:
        raise ValueError(f"Unexpected SHAP shape for {key}: {shap_values.shape} vs {X_test.shape}")
    shap_results[key] = {
        "explainer": explainer,
        "values": shap_values,
        "X_test": X_test.copy(),
        "dates_test": result["split"]["dates_test"].reset_index(drop=True),
    }
    print(f"SHAP OK {commodity}/{target}: values={shap_values.shape}")

print(f"Computed SHAP for {len(shap_results)} models.")

In [ ]:
# Per-commodity feature importance plots

def feature_color(feature_name: str) -> str:
    return "#1f77b4" if any(feature_name.startswith(f"{cp}_") for cp in CHOKEPOINTS) else "#ff7f0e"

fig, axes = plt.subplots(nrows=len(COMMODITIES), ncols=len(TARGETS), figsize=(18, 4 * len(COMMODITIES)))
if len(COMMODITIES) == 1:
    axes = np.array([axes])

for row_idx, commodity in enumerate(COMMODITIES):
    for col_idx, target in enumerate(TARGETS):
        ax = axes[row_idx, col_idx]
        key = (commodity, target)
        values = shap_results[key]["values"]
        mean_abs = pd.Series(np.abs(values).mean(axis=0), index=FEATURE_COLS).sort_values(ascending=False).head(15)
        colors = [feature_color(name) for name in mean_abs.index]
        ax.barh(mean_abs.index[::-1], mean_abs.values[::-1], color=colors[::-1])
        ax.set_title(f"{commodity} — {target}")
        ax.set_xlabel("mean |SHAP|")

fig.suptitle("Top 15 features per commodity per target", fontsize=18, y=1.01)
plt.tight_layout()
plt.show()
print("Rendered per-commodity SHAP feature importance grid.")

In [ ]:
# Aggregate attribution matrix
attribution_rows = []

for target in TARGETS:
    for commodity in COMMODITIES:
        key = (commodity, target)
        values = shap_results[key]["values"]
        mean_abs_by_feature = pd.Series(np.abs(values).mean(axis=0), index=FEATURE_COLS)
        total_chokepoint_shap = 0.0
        cp_values = {}
        for cp in CHOKEPOINTS:
            cp_total = float(mean_abs_by_feature[[c for c in FEATURE_COLS if c.startswith(f"{cp}_")]].sum())
            cp_values[cp] = cp_total
            total_chokepoint_shap += cp_total
        for cp, cp_total in cp_values.items():
            attribution_rows.append({
                "chokepoint": cp,
                "commodity": commodity,
                "target_horizon": target,
                "mean_abs_shap": cp_total,
                "normalized_share": cp_total / total_chokepoint_shap if total_chokepoint_shap else np.nan,
            })

attribution_df = pd.DataFrame(attribution_rows)

for target in TARGETS:
    subset = attribution_df[attribution_df["target_horizon"] == target]
    raw = subset.pivot(index="chokepoint", columns="commodity", values="mean_abs_shap").loc[CHOKEPOINTS, COMMODITIES]
    normalized = subset.pivot(index="chokepoint", columns="commodity", values="normalized_share").loc[CHOKEPOINTS, COMMODITIES]
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.heatmap(raw, annot=True, fmt=".3f", cmap="Blues", ax=axes[0])
    axes[0].set_title(f"Chokepoint attribution — {target} raw")
    sns.heatmap(normalized, annot=True, fmt=".1%", cmap="Oranges", ax=axes[1])
    axes[1].set_title(f"Chokepoint attribution — {target} column-normalized")
    plt.tight_layout()
    plt.show()

print("Attribution rows:", len(attribution_df))
display(attribution_df.head())

In [ ]:
# Save artifacts to HDFS for dashboard integration
written_artifacts: list[dict[str, object]] = []


def write_parquet_and_upload(df: pd.DataFrame, local_path: Path, hdfs_path: str) -> None:
    """Write a local copy for debugging, then upload bytes through HttpFS."""
    local_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(local_path, index=False)
    size = hdfs_upload_v2(local_path, hdfs_path, overwrite=True)
    written_artifacts.append({"hdfs_path": hdfs_path, "bytes": size})

for key, result in results.items():
    commodity, target = key
    safe_target = target
    pred_df = result["predictions"].copy()
    pred_hdfs = f"/supply-chain/analytics/predictions_v2/{commodity}_{safe_target}/{commodity}_{safe_target}.parquet"
    pred_local = LOCAL_ARTIFACT_ROOT / "predictions_v2" / f"{commodity}_{safe_target}.parquet"
    write_parquet_and_upload(pred_df, pred_local, pred_hdfs)

    shap_info = shap_results[key]
    X_test = shap_info["X_test"].reset_index(drop=True)
    values = shap_info["values"]
    dates = shap_info["dates_test"].reset_index(drop=True)
    shap_frames = []
    for feature_idx, feature_name in enumerate(FEATURE_COLS):
        shap_frames.append(pd.DataFrame({
            "event_date": dates,
            "feature_name": feature_name,
            "shap_value": values[:, feature_idx],
            "feature_value": X_test[feature_name].to_numpy(),
        }))
    shap_local_df = pd.concat(shap_frames, ignore_index=True)
    shap_hdfs = f"/supply-chain/analytics/shap_local_v2/{commodity}_{safe_target}/{commodity}_{safe_target}.parquet"
    shap_local = LOCAL_ARTIFACT_ROOT / "shap_local_v2" / f"{commodity}_{safe_target}.parquet"
    write_parquet_and_upload(shap_local_df, shap_local, shap_hdfs)

metrics_out = metrics_df[["commodity", "target", "test_r2", "test_mae", "naive_mae", "improvement_pct", "spearman", "decision_flag"]].copy()
metrics_hdfs = "/supply-chain/analytics/regression_metrics/metrics.parquet"
metrics_local = LOCAL_ARTIFACT_ROOT / "regression_metrics" / "metrics.parquet"
write_parquet_and_upload(metrics_out, metrics_local, metrics_hdfs)

attribution_hdfs = "/supply-chain/analytics/shap_attribution_v2/attribution.parquet"
attribution_local = LOCAL_ARTIFACT_ROOT / "shap_attribution_v2" / "attribution.parquet"
write_parquet_and_upload(attribution_df, attribution_local, attribution_hdfs)

metadata = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "model_decision_verdict": VERDICT,
    "average_test_r2": avg_test_r2,
    "models_with_improvement_gt_5pct": models_improving,
    "feature_contract_version": "rolling_gdelt_96_macro_vol_targets_v2",
    "feature_columns": FEATURE_COLS,
    "feature_column_count": len(FEATURE_COLS),
    "target_definitions": {
        "abs_return_5d_fwd": "abs((price[t+5] / price[t]) - 1)",
        "abs_return_20d_fwd": "abs((price[t+20] / price[t]) - 1)",
    },
    "split_logic": "Sort by event_date; first 70% train, next 15% validation, final 15% test; no shuffle.",
    "decision_criteria": {
        "average_test_r2_gt": 0.10,
        "min_models_with_improvement_pct_gt_5": 6,
    },
}
metadata_local = LOCAL_ARTIFACT_ROOT / "regression_metadata.json"
metadata_local.write_text(json.dumps(metadata, indent=2))
metadata_hdfs = "/supply-chain/analytics/regression_metadata.json"
metadata_size = hdfs_upload_v2(metadata_local, metadata_hdfs, overwrite=True)
written_artifacts.append({"hdfs_path": metadata_hdfs, "bytes": metadata_size})

print("Artifacts written to HDFS:")
for artifact in written_artifacts:
    print(f"{artifact['bytes']:>10,} bytes  {artifact['hdfs_path']}")

# Summary

The final model decision verdict is printed as `DECISION_VERDICT` in the decision gate cell and persisted to `/supply-chain/analytics/regression_metadata.json` as `model_decision_verdict`.

Artifacts written by this notebook:

- `/supply-chain/analytics/predictions_v2/{commodity}_{target}/{commodity}_{target}.parquet`
- `/supply-chain/analytics/shap_local_v2/{commodity}_{target}/{commodity}_{target}.parquet`
- `/supply-chain/analytics/regression_metrics/metrics.parquet`
- `/supply-chain/analytics/shap_attribution_v2/attribution.parquet`
- `/supply-chain/analytics/regression_metadata.json`

If `USABLE`, the dashboard rebuild will integrate predictions and SHAP via the new HDFS paths. If `NOT_USABLE`, the dashboard ships query-driven without ML predictions.